# Open-Ended INCLUDE Probability Windows: Subset

Compare raw LogitLens top-p and Repr-GMM for Llama 2, OLMo 2, Aya-23, and Apertus.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vis import (
    overlay_story_category_max_markers,
    plot_story_category_bar_grid,
    save_matplotlib_figure_bundle,
    set_matplotlib_paper_font,
)

CACHE_DIR = REPO_ROOT / ".analysis_cache" / "openended_language_probs_layer_windows"
LANGDIST_CACHE_PATH = CACHE_DIR / "combined_late_layer_langdist_by_language.parquet"
MAX_OTHER_CACHE_PATH = CACHE_DIR / "combined_late_layer_max_other_langdist.parquet"
FIG_DIR = REPO_ROOT / "figs" / "subsets"

METHOD_PANEL_MAP = {
    "raw-rtopp": "Raw Logitlens Top-p",
    "repr": "Repr-GMM",
}
PANEL_ORDER = list(METHOD_PANEL_MAP.values())
MODEL_ORDER = [
    "Llama-2-7B",
    "OLMo-2-1124-7B",
    "Aya-23-8B",
    "Apertus-8B",
]
MODEL_LABEL_MAP = {
    "Llama-2-7B": "Llama2",
    "OLMo-2-1124-7B": "OLMo2",
    "Aya-23-8B": "Aya-23",
    "Apertus-8B": "Apertus",
}
LAYER_ROW_MAP = {
    "50%-75% Layers": "Layers\n50%-75%",
    "75%-100% Layers": "Layers\n75%-100%",
}
ROW_ORDER = list(LAYER_ROW_MAP.values())
CATEGORY_ORDER = ["task_relevant", "english", "other"]
CATEGORY_LABELS = {
    "task_relevant": "Avg. P(L=task-relevant)",
    "english": "P(L=English)",
    "other": "Avg. P(L=other)",
}
CATEGORY_COLORS = {
    "task_relevant": "#2f7ed8",
    "english": "#d95f02",
    "other": "#999999",
}

set_matplotlib_paper_font()


In [ ]:
def summarize_language_aggregates(aggregate_df):
    """Convert cached probability moments into means and standard errors."""
    group_cols = [
        "method_label",
        "display_model_name",
        "layer_window",
        "prob_category",
    ]
    summary = aggregate_df.groupby(group_cols, dropna=False).agg(
        sum_prob=("sum_prob", "sum"),
        n_values=("n_values", "sum"),
        sum_sq_prob=("sum_sq_prob", "sum"),
    ).reset_index()
    summary["mean_prob"] = summary["sum_prob"] / summary["n_values"].clip(lower=1)
    variance_numerator = (
        summary["sum_sq_prob"]
        - summary["sum_prob"].pow(2) / summary["n_values"].clip(lower=1)
    ).clip(lower=0.0)
    summary["var_prob"] = variance_numerator / (summary["n_values"] - 1).clip(lower=1)
    summary.loc[summary["n_values"] <= 1, "var_prob"] = 0.0
    summary["stderr_prob"] = np.sqrt(summary["var_prob"]) / np.sqrt(
        summary["n_values"].clip(lower=1)
    )
    return summary


for cache_path in (LANGDIST_CACHE_PATH, MAX_OTHER_CACHE_PATH):
    if not cache_path.exists():
        raise FileNotFoundError(
            f"Missing {cache_path}. Build the open-ended aggregation caches first."
        )

language_aggregates = pd.read_parquet(LANGDIST_CACHE_PATH)
max_other_aggregates = pd.read_parquet(MAX_OTHER_CACHE_PATH)
selection = (
    language_aggregates["method_label"].isin(METHOD_PANEL_MAP)
    & language_aggregates["display_model_name"].isin(MODEL_ORDER)
)
bar_summary = summarize_language_aggregates(language_aggregates[selection].copy())
bar_summary = bar_summary[bar_summary["prob_category"].isin(CATEGORY_ORDER)].copy()

max_other_selection = (
    max_other_aggregates["method_label"].isin(METHOD_PANEL_MAP)
    & max_other_aggregates["display_model_name"].isin(MODEL_ORDER)
)
max_other_summary = summarize_language_aggregates(
    max_other_aggregates[max_other_selection].copy()
)
max_other_summary["prob_category"] = "other"
max_task_summary = bar_summary[bar_summary["prob_category"].eq("task_relevant")].copy()
marker_summary = pd.concat([max_task_summary, max_other_summary], ignore_index=True)

for summary in (bar_summary, marker_summary):
    summary["panel"] = summary["method_label"].map(METHOD_PANEL_MAP)
    summary["layer_row"] = summary["layer_window"].map(LAYER_ROW_MAP)

available_pairs = set(zip(bar_summary["display_model_name"], bar_summary["method_label"]))
expected_pairs = {(model, method) for model in MODEL_ORDER for method in METHOD_PANEL_MAP}
missing_pairs = sorted(expected_pairs - available_pairs)
if missing_pairs:
    raise ValueError(f"Missing requested model/method rows: {missing_pairs}")


In [ ]:
fig = plot_story_category_bar_grid(
    bar_summary,
    row_col="layer_row",
    row_order=ROW_ORDER,
    col_col="panel",
    col_order=PANEL_ORDER,
    models=MODEL_ORDER,
    category_order=CATEGORY_ORDER,
    category_labels=CATEGORY_LABELS,
    category_colors=CATEGORY_COLORS,
    probability_label="Average Language Probability",
    shared_ylabel="Avg. Lang Prob across Prompts and Layers",
    show_row_label_in_ylabel=True,
    title=None,
    subtitle=None,
    unavailable_label="open-ended INCLUDE subset",
    figsize=(10.5, 6.4),
    legend_y=0.015,
    bottom=0.265,
    top=0.860,
    left=0.190,
    right=0.990,
    shared_ylabel_x=-0.010,
    sharey=True,
    model_label_map=MODEL_LABEL_MAP,
)
if fig is not None:
    overlay_story_category_max_markers(
        fig,
        bar_summary,
        marker_summary,
        row_col="layer_row",
        col_col="panel",
        row_order=ROW_ORDER,
        col_order=PANEL_ORDER,
        models=MODEL_ORDER,
        category_order=CATEGORY_ORDER,
        category_colors=CATEGORY_COLORS,
        legend_y=0.015,
        legend_fontsize=16,
        x_edge_padding=0.46,
        x_tick_shift_points=14.0,
    )
    panel_axes = np.asarray(fig.axes[:4], dtype=object).reshape(2, 2)
    fig.subplots_adjust(hspace=0.035, wspace=0.025)
    for col_idx, panel_name in enumerate(PANEL_ORDER):
        panel_axes[0, col_idx].set_title(
            panel_name, fontsize=18, pad=9, fontweight="bold"
        )
    for ax in panel_axes.ravel():
        ax.tick_params(axis="both", labelsize=15)
        ax.xaxis.label.set_size(16)
        ax.yaxis.label.set_size(17)
    for row_ax in panel_axes[:, 0]:
        row_ax.yaxis.label.set_size(18)
        row_ax.yaxis.label.set_fontweight("bold")
        row_ax.yaxis.set_label_coords(-0.2, 0.5)
    for label_ax in panel_axes[-1, :]:
        label_ax.tick_params(axis="x", labelsize=17)
    if getattr(fig, "_supylabel", None) is not None:
        panel_y0 = min(ax.get_position().y0 for ax in panel_axes.ravel())
        panel_y1 = max(ax.get_position().y1 for ax in panel_axes.ravel())
        fig._supylabel.set_position((0.02, (panel_y0 + panel_y1) / 2.0))
        fig._supylabel.set_fontsize(18)
    panel_top = max(ax.get_position().y1 for ax in panel_axes.ravel())
    panel_left = panel_axes[0, 0].get_position().x0
    panel_right = panel_axes[0, -1].get_position().x1
    dataset_label_y = panel_top + 0.062
    fig.text(
        (panel_left + panel_right) / 2.0,
        dataset_label_y,
        "Open-ended: INCLUDE",
        ha="center",
        va="bottom",
        fontsize=18,
        fontweight="bold",
    )
    fig.add_artist(plt.Line2D(
        [panel_left + 0.025 * (panel_right - panel_left), panel_right - 0.025 * (panel_right - panel_left)],
        [dataset_label_y - 0.007, dataset_label_y - 0.007],
        transform=fig.transFigure,
        color="#222222",
        linewidth=0.9,
        alpha=0.85,
    ))
    save_matplotlib_figure_bundle(
        fig,
        FIG_DIR / "openended_include_probability_windows_subset",
    )
    plt.show()
